Pandas 常以 NaN、NaT 或`<NA>` 表示缺失值。

缺失不一定代表「資料錯了」，它可能來自很多情況，例如：

- 使用者忘記填寫
- 系統沒有成功取得資料
- 該欄位不適用
- 使用者拒絕提供
- 資料匯入時格式錯誤

所以看到缺失值時，不是立刻全部刪掉，而是先判斷這個欄位對分析有多重要。

```py
df.isna() # 檢查每個位置是否為缺失值
df.isna().sum() # 統計每個欄位有多少個缺失值
```

### 策略一：刪除缺失資料

有些欄位不能缺。

例如「交易編號」如果不存在，後續就很難辨認這筆交易，也可能無法和其他資料表進行 merge。這種情況可以直接刪除。

In [142]:
import pandas as pd


### 建立範例資料
data = {
    "交易編號": ["T001", "T002", None, "T004"],
    "類別": ["餐飲", "交通", "購物", "餐飲"],
    "金額": [120, 250, 300, 180]
}

df = pd.DataFrame(data)

### 刪除交易編號缺失的資料
df = df.dropna(
    subset=["交易編號"]
) # 只檢查「交易編號」欄位，缺失就刪除整列

df

,交易編號,類別,金額
0,T001,餐飲,120
1,T002,交通,250
3,T004,餐飲,180


### 策略二：填補缺失值

有些資料雖然缺失，但不需要直接刪掉。

In [143]:
import pandas as pd


### 建立範例資料
data = {
    "交易編號": ["T001", "T002", "T003", "T004"],
    "類別": ["餐飲", None, "交通", None],
    "金額": [120, 250, None, 180]
}

df = pd.DataFrame(data)

### 填補缺失值
df["類別"] = df["類別"].fillna(
    "未分類"
) # 將「類別」缺失的位置填成「未分類」

df

,交易編號,類別,金額
0,T001,餐飲,120.0
1,T002,未分類,250.0
2,T003,交通,NaN
3,T004,未分類,180.0


In [144]:
# 數值資料也可以填補。
# 例如有一筆「金額」缺失，就用金額中位數去補
median_amount = df["金額"].median() # 計算金額中位數

df["金額"] = df["金額"].fillna(
    median_amount
) # 使用中位數填補缺失金額

df


,交易編號,類別,金額
0,T001,餐飲,120.0
1,T002,未分類,250.0
2,T003,交通,180.0
3,T004,未分類,180.0


### 策略三：保留缺失值，另外建立標記

有時候「缺失」本身就是資訊。

例如「金額沒有填」，可能代表：

- 系統取得資料失敗
- 使用者沒有提供
- 某種特殊交易沒有金額
- 資料串接發生問題

如果直接把缺失值填掉，這些資訊就消失了。這時可以建立一個新的欄位，記錄原本是否缺失。

In [145]:
import pandas as pd


### 建立範例資料
data = {
    "交易編號": ["T001", "T002", "T003", "T004"],
    "類別": ["餐飲", "交通", "購物", "餐飲"],
    "金額": [120, None, 300, None]
}

df = pd.DataFrame(data)


### 建立缺失標記欄位
df["金額是否缺失"] = df["金額"].isna()

df

,交易編號,類別,金額,金額是否缺失
0,T001,餐飲,120.0,False
1,T002,交通,NaN,True
2,T003,購物,300.0,False
3,T004,餐飲,NaN,True


### Problem. 

In [146]:
import pandas as pd


### 建立訂單資料
data = {
    "訂單編號": [
        "A001",
        "A002",
        None,
        "A004",
        "A005",
        "A006",
        "A007",
        "A008"
    ],
    "餐點類別": [
        "便當",
        "飲料",
        "麵食",
        None,
        "便當",
        "飲料",
        None,
        "麵食"
    ],
    "餐點金額": [
        120,
        80,
        150,
        200,
        None,
        90,
        None,
        180
    ],
    "付款方式": [
        "信用卡",
        "現金",
        "信用卡",
        "Line Pay",
        "現金",
        None,
        "信用卡",
        "現金"
    ]
}


### 建立 DataFrame
df = pd.DataFrame(data)


### 顯示原始資料
print("原始資料：")
df

原始資料：


,訂單編號,餐點類別,餐點金額,付款方式
0,A001,便當,120.0,信用卡
1,A002,飲料,80.0,現金
2,NaN,麵食,150.0,信用卡
3,A004,NaN,200.0,Line Pay
4,A005,便當,NaN,現金
5,A006,飲料,90.0,NaN
6,A007,NaN,NaN,信用卡
7,A008,麵食,180.0,現金


### Problem. 檢查 DataFrame 哪些位置存在缺失值?

In [147]:
df.isna()

,訂單編號,餐點類別,餐點金額,付款方式
0,False,False,False,False
1,False,False,False,False
2,True,False,False,False
3,False,True,False,False
4,False,False,True,False
5,False,False,False,True
6,False,True,True,False
7,False,False,False,False


### Problem. 統計每個欄位分別有幾個缺失值。

In [148]:
df.isna().sum()

訂單編號    1
餐點類別    2
餐點金額    2
付款方式    1
dtype: int64

### Problem. 

「訂單編號」是辨認一筆訂單的重要欄位。如果訂單編號不存在，我們決定直接刪除該筆資料。

In [149]:
df.dropna(subset=["訂單編號"])
df

,訂單編號,餐點類別,餐點金額,付款方式
0,A001,便當,120.0,信用卡
1,A002,飲料,80.0,現金
2,NaN,麵食,150.0,信用卡
3,A004,NaN,200.0,Line Pay
4,A005,便當,NaN,現金
5,A006,飲料,90.0,NaN
6,A007,NaN,NaN,信用卡
7,A008,麵食,180.0,現金


### Problem. 

部分訂單沒有餐點類別，但我們不希望因此刪掉整筆訂單。

請將「餐點類別缺失值」統一填成「未分類」

In [150]:
df["餐點類別"] = df["餐點類別"].fillna("未分類")
df

,訂單編號,餐點類別,餐點金額,付款方式
0,A001,便當,120.0,信用卡
1,A002,飲料,80.0,現金
2,NaN,麵食,150.0,信用卡
3,A004,未分類,200.0,Line Pay
4,A005,便當,NaN,現金
5,A006,飲料,90.0,NaN
6,A007,未分類,NaN,信用卡
7,A008,麵食,180.0,現金


### Problem. 

現在發現部分訂單的「餐點金額」沒有資料。請用有效餐點金額的中位數填補。


In [151]:
median = df["餐點金額"].median()
df["餐點金額"] = df["餐點金額"].fillna(median)
df

,訂單編號,餐點類別,餐點金額,付款方式
0,A001,便當,120.0,信用卡
1,A002,飲料,80.0,現金
2,NaN,麵食,150.0,信用卡
3,A004,未分類,200.0,Line Pay
4,A005,便當,135.0,現金
5,A006,飲料,90.0,NaN
6,A007,未分類,135.0,信用卡
7,A008,麵食,180.0,現金


### Problem. 

A006 的「付款方式」沒有資料。請用眾數填補。

In [152]:
mode = df["付款方式"].mode()[0]
df["付款方式"] = df["付款方式"].fillna(mode)
df

,訂單編號,餐點類別,餐點金額,付款方式
0,A001,便當,120.0,信用卡
1,A002,飲料,80.0,現金
2,NaN,麵食,150.0,信用卡
3,A004,未分類,200.0,Line Pay
4,A005,便當,135.0,現金
5,A006,飲料,90.0,信用卡
6,A007,未分類,135.0,信用卡
7,A008,麵食,180.0,現金
